# Interview Coach Agent — 실행 데모 (G1)

실제 서버(`server.py`)를 sample_user 데이터·`openai:gpt-4o-mini`로 띄우고, 브라우저가 쓰는 것과 같은 HTTP API로 E2E를 실행한 기록이다.
시나리오: ① 문장 표현 카드 → 새로고침 복원 → 승인·저장 ② 면접 질문 답변 ③ 복습 ④ 오류 처리(범위 밖·중복 승인·준비 미완료).
흐름 설명은 `doc/handoff/flow-guide.md`.

In [1]:
import json, os, shutil, sys, tempfile, httpx
from pathlib import Path
from dotenv import load_dotenv
os.environ["EVAL_PORT"] = "8140"
sys.path.insert(0, ".")
from eval.run_v2 import BASE, start_server
load_dotenv(".env")
work = Path(tempfile.mkdtemp(prefix="demo_")); data = work / "data"
shutil.copytree("examples/sample_user", data)
server = start_server(data, "openai:gpt-4o-mini")
c = httpx.Client(base_url=BASE, timeout=180)
print("서버:", BASE)
for item in c.get("/api/readiness").json()["items"]:
    print(f"  준비 점검 {item['item']}: {item['status']} — {item['message']}")

서버: http://127.0.0.1:8140
  준비 점검 프로필: ok — 이커머스 데이터 분석가 (신입) / 자격증 1개
  준비 점검 경험 문서: ok — docs/ 1개 · 활동·문항 3개 · 1,115자
  준비 점검 공고: ok — 필수 섹션 2개 모두 있음
  준비 점검 개인정보: warn — 전화번호·이메일 패턴은 없지만, 외부 LLM 사용 시 서류가 전송됩니다


In [2]:
LABEL = {"RouteDecided": "분류", "EvidenceSearched": "근거 검색", "CardDrafted": "초안", "ReviewDone": "검수",
         "ApprovalRequired": "승인 대기", "FinalAnswer": "답변", "ErrorRaised": "오류"}

def chat(message, thread_id):
    events = []
    with c.stream("GET", "/chat/stream", params={"message": message, "thread_id": thread_id}) as r:
        for line in r.iter_lines():
            if line.startswith("data: ") and line != "data: [DONE]":
                e = json.loads(line[6:]); events.append(e)
                detail = {"RouteDecided": e.get("intent"), "EvidenceSearched": f"{e.get('tool')} '{e.get('query')}'",
                          "CardDrafted": f"{e.get('attempt')}회차", "ReviewDone": f"{e.get('round')}라운드 통과={e.get('passed')} " + "; ".join(f"{v['rule_id']} '{v['quote'][:30]}'" for v in e.get("violations", [])),
                          "FinalAnswer": (e.get("reply") or "")[:120], "ErrorRaised": (e.get("error") or {}).get("code")}.get(e["type"], "")
                print(f"  [{LABEL[e['type']]}] {detail}")
    print("  [DONE]")
    return events

def show_card(ev):
    approval = next(e for e in ev if e["type"] == "ApprovalRequired"); card = approval["card"]
    if approval.get("warning"):
        print("  ⚠ 경고:", approval["warning"][:160])
    print(f"  Formal : {card['formal_expression']}\n  Casual : {card['casual_expression']}")
    if card.get("answer_draft"):
        print(f"  답변 초안: {card['answer_draft']}")
    print("  핵심 표현:", ", ".join(ch["phrase"] for ch in card["chunks"]))
    print("  근거:", card["evidence"])

In [3]:
print("① 문장 표현"); ev = chat("배송 지연의 61%가 특정 택배사의 집하 구간에 몰려 있다는 걸 찾았어요", "demo-1")
show_card(ev)
print("\n새로고침 → GET /chat/pending")
p = c.get("/chat/pending", params={"thread_id": "demo-1"}).json()
print("  승인 대기 복원:", p["pending"] is not None, "/", p["pending"]["card"]["formal_expression"][:70])
print("\n승인 → POST /chat/resume:", c.post("/chat/resume", json={"thread_id": "demo-1", "approve": True}).json())

① 문장 표현


  [분류] express


  [근거 검색] search_my_experience '배송 지연 분석 결과'


  [근거 검색] search_expression_bank '데이터 분석 결과 제시'


  [초안] 1회차


  [검수] 1라운드 통과=True 
  [승인 대기] 
  [DONE]
  Formal : I found that 61% of delivery delays were concentrated in the pickup segment of a specific courier company.
  Casual : I discovered that 61% of the delivery delays were all coming from one courier's pickup area.
  핵심 표현: 61% of delivery delays, concentrated in the pickup segment, a specific courier company
  근거: ['experience.md > 물류 스타트업 데이터 분석 인턴 (2개월)']

새로고침 → GET /chat/pending
  승인 대기 복원: True / I found that 61% of delivery delays were concentrated in the pickup se

승인 → POST /chat/resume: {'saved': True, 'card_id': 'C0001', 'next_review': '2026-10-22'}


In [4]:
print("② 면접 질문 답변"); ev = chat("면접 질문: 분석 결과가 현업의 예상과 다를 때 어떻게 설득하겠습니까?", "demo-2")
show_card(ev)
print("\n승인:", c.post("/chat/resume", json={"thread_id": "demo-2", "approve": True}).json())

② 면접 질문 답변


  [분류] answer


  [근거 검색] get_job_posting '자격 요건'
  [근거 검색] search_expression_bank '결과를 다루는 법'


  [근거 검색] search_my_experience '이탈 예측 모델 분석 결과'


  [초안] 1회차


  [검수] 1라운드 통과=False R3 'I believe it's important to fi'; R8 'This led me to adjust our eval'; R6 'achieving a 2.3x increase on t'


  [근거 검색] get_job_posting '자격 요건'
  [근거 검색] search_my_experience '분석 결과 예상과 다를 때 팀과의 협력 경험'


  [초안] 2회차


  [검수] 2라운드 통과=True 
  [승인 대기] 
  [DONE]
  Formal : When the analysis results differ from expectations, I first check in with the team to understand their perspective. During my project on customer churn prediction, I realized the importance of considering different costs associated with prediction errors, after feedback from my professor. This collaborative approach helped us focus our coupon targeting strategy, which allowed us to capture a significant portion of at-risk customers effectively. I would apply a similar method here, ensuring we all have aligned understanding before moving forward with decisions.
  Casual : If my analysis results don't match expectations, I'd first check in with the team to see what they think. In my churn prediction project, I learned that we needed to consider different costs when predictions went wrong, thanks to my professor's input. This teamwork helped us target our coupons better and reach more at-risk customers. I’d do something similar here to m

In [5]:
print("③ 복습"); chat("오늘 복습할 카드 보여줘", "demo-3")
print("\n카드함 GET /api/cards:")
for card in c.get("/api/cards").json():
    print(f"  {card['id']} 다음 복습 {card['next_review']} 단계 {card['stage']} — {card['korean_prompt'][:40]}")

③ 복습


  [분류] review
  [답변] 오늘 복습할 카드는 없어요. 최근 저장한 카드 2장을 보여 드릴게요.

[C0002] 분석 결과가 현업의 예상과 다를 때 어떻게 설득하겠습니까?
  ↳ 정답: When the analysis results diffe
  [DONE]

카드함 GET /api/cards:
  C0001 다음 복습 2026-10-22 단계 0 — 배송 지연의 61%가 특정 택배사의 집하 구간에 몰려 있다는 걸 찾았어요
  C0002 다음 복습 2026-10-22 단계 0 — 분석 결과가 현업의 예상과 다를 때 어떻게 설득하겠습니까?


In [6]:
print("④-1 범위 밖 요청"); chat("오늘 점심 메뉴 추천해줘", "demo-4")
print("\n④-2 이미 승인한 카드를 다시 승인 → 409")
r = c.post("/chat/resume", json={"thread_id": "demo-1", "approve": True}); print(" ", r.status_code, r.json()["error"]["code"], r.json()["error"]["user_message"])
print("\n④-3 공고가 없을 때 질문 → SSE ErrorRaised(NOT_READY)")
(data / "job_posting.md").rename(work / "posting.bak")
chat("고객 이탈을 미리 예측하는 모델을 만들었어요", "demo-5")
(work / "posting.bak").rename(data / "job_posting.md")

④-1 범위 밖 요청


  [분류] out_of_scope
  [답변] 저는 영어 면접 표현·답변 준비와 복습만 도와드려요. 면접에서 하고 싶은 말이나 면접 질문을 입력해 주세요.
  [DONE]

④-2 이미 승인한 카드를 다시 승인 → 409
  409 NO_PENDING 승인 대기 중인 카드가 없어요. 이미 처리됐을 수 있어요.

④-3 공고가 없을 때 질문 → SSE ErrorRaised(NOT_READY)
  [오류] NOT_READY
  [DONE]


PosixPath('/var/folders/t6/j2yqpqpn0h3_mys7lr5tkgqc0000gn/T/demo_mof7_tql/data/job_posting.md')

In [7]:
server.terminate(); server.wait(timeout=30); print("서버 종료")

서버 종료
